# CPER Soil-Moisture — Pegasus Workflow

Runs the CPER soil-moisture pipeline as a **Pegasus WMS workflow** on an
HTCondor pool: generate the DAG, inspect it, plan and submit, monitor, then
look at the staged results.

What Pegasus buys here, concretely:

* **Fan-out.** NEON serves one package per site-month, so a decade-long
  characterization becomes ~120 monthly fetch jobs running *in parallel*
  instead of one job walking 120 site-months serially.
* **Retry.** Station APIs are flaky; each fetch job carries a DAGMan retry.
* **Data reuse.** The expensive static branch (soil + terrain covariates) is
  registered in the replica catalog and skipped by later runs — which is what
  makes a repeating nowcast cheap enough to call "dynamic".
* **Provenance.** Every product traces back to the run record and the
  covariate manifest it was built from.

To run the same stages on a laptop with no Pegasus at all, see
**`Run-CPER-SoilMoisture-Locally.ipynb`**.

### What the workflow answers

The CPER researcher asked for three things, in order:

| # | Ask | Stages |
|---|-----|--------|
| 1 | Characterize each station's historical soil-moisture response, relate it to soil/topography/climate, and visualize it | `station_response` → `similarity_cluster` → `attribute` ×9 → `similarity_merge` → `visualize_response` |
| 2 | Delineate areas expected to behave similarly | `delineate_zones` |
| 3 | Estimate current soil moisture site-wide, updating as new data arrive | `soil_moisture_map` (point scale) → `estimate_soil_moisture` → `visualize_soil_moisture` |

All three are built, and the pipeline has run end to end on a real HTCondor
pool: 186 executable jobs, 9 min 56 s, zero failures.

`SPEC.md` is the single design document; `README.md` has the command-line
equivalent of everything here.

## 0. Setup and prerequisites

Workflow *generation* needs only `pegasus-wms.api` (a pip package).
*Submission* needs a Pegasus install and a configured HTCondor pool. The cells
below are safe to run without one — they report what is missing rather than
failing.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

import pandas as pd

# Submit hosts are often older than a laptop: ACCESS Pegasus ships Python 3.6,
# where subprocess.run has neither capture_output nor text (both 3.7+). Use the
# portable spelling so these notebooks run there unchanged.
def sh(cmd):
    # Returns (returncode, combined stdout+stderr).
    p = subprocess.run([str(c) for c in cmd], stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, universal_newlines=True)
    return p.returncode, p.stdout or ""


WF = Path.cwd()                      # notebook lives in the workflow root
BIN = WF / "bin"
CONFIG = WF / "site_config.json"

assert BIN.exists(), f"run this notebook from the workflow root; got {WF}"
print("python :", sys.version.split()[0])
print("workflow root:", WF)

import shutil

HAVE_PLAN = shutil.which("pegasus-plan") is not None
HAVE_CONDOR = shutil.which("condor_status") is not None
print("pegasus-plan  :", shutil.which("pegasus-plan") or "NOT FOUND")
print("condor_status :", shutil.which("condor_status") or "NOT FOUND")

try:
    import Pegasus.api  # noqa: F401
    print("Pegasus.api   : available (generation will work)")
except ImportError:
    print("Pegasus.api   : MISSING - pip install pegasus-wms.api")

if not HAVE_PLAN:
    print("\nNo Pegasus CLI here: you can still generate and inspect the DAG,")
    print("then copy it to a submit host to run.")

### Run configuration

Fingerprinting needs a long record, so the default window is multi-year. SCAN
and USCRN are anonymous; **NEON needs a free token**
(data.neonscience.org → My Account → API Tokens). Without it the NEON source
is skipped and the run continues on SCAN — the same graceful degradation the
workflow uses on a pool.

USCRN is off by default: `www.ncei.noaa.gov` has been unreachable from several
networks, and the on-site Nunn station stopped publishing on 2026-05-28, so
enabling it mostly buys ~35 minutes of retries.

In [ ]:
START = "2014-01-01"                 # long enough for real fingerprints
END   = str(date.today())

# Optional: paste a NEON token here, or export NEON_TOKEN before launching.
NEON_TOKEN = os.environ.get("NEON_TOKEN", "")

SOURCES = ["awdb"]                   # SCAN: anonymous, 5 depths, back to 1997
if NEON_TOKEN:
    SOURCES.append("neon")           # 5 soil plots, ~38 depth-nodes
    os.environ["NEON_TOKEN"] = NEON_TOKEN

INCLUDE_USCRN = False
if INCLUDE_USCRN:
    SOURCES.insert(0, "uscrn")

print("window :", START, "->", END)
print("sources:", ", ".join(SOURCES))
print("NEON   :", "token present" if NEON_TOKEN else "no token - NEON skipped")

## 1. Configure the run

| mode | what it builds |
|---|---|
| **`all`** | **the default** — the whole pipeline on the historical window, ending in the gridded dynamic map. 147 abstract jobs, 186 executable after planning. |
| `nowcast` | the same DAG on the last 30 days. Meant to be paired with reuse, which reduces it to ~16 executable jobs. |
| `characterize` | everything except the gridded map — observations, covariates, fingerprints, zones and attribution. |
| `static` | soil + terrain → covariate stack, **registered** for reuse by later runs. |
| `observe` | observations only — fetch, harmonize, point-scale layer. |
| `fetch` | download only: every fetcher, outputs staged out and registered. |

This notebook runs **two workflows**: a `fetch` run that fills `inputs/`, then
the processing run that consumes it. Splitting them is what makes the second
one repeatable — re-run the analysis as often as you like against the same
downloads, without paying for 121 NEON site-months again.

⚠ **`nowcast` without reuse is a trap.** It would recompute the response
fingerprints from the 30-day window, so the "climatology" each anomaly is
measured against becomes that same window and the map collapses toward
climatology. The generator warns if you do it.

`NEON_TOKEN` must be set **before building**: the generator captures it and
injects it into the job environment with `add_env`, because an HTCondor job
never sees the submit shell's exports. This is the single most common reason
a NEON job comes back empty.

In [ ]:
MODE = "all"                 # the full pipeline, ending in the gridded map
SITE = "condorpool"          # execution site name in your Pegasus site catalog
INPUTS = WF / "inputs"       # downloads land here; results go to ./output

# Run the fetch workflow below? Set False if inputs/ is already populated --
# by an earlier run of this notebook, or by ./fetch_data.sh, which does the
# same downloads without a pool and is faster on a submit host (4 m 23 s vs
# 7 m 18 s, because 167 jobs each pay scheduling overhead to wrap a ~2 s
# download).
RUN_FETCH = True

# HTCondor universe for the execution site. "container" (the default) has
# HTCondor create the container and run PegasusLite inside it — what Pegasus
# recommends for every HTCondor pool, and the only thing that works where the
# execution point is itself an unprivileged container (OSG / OSPool / PATh, i.e.
# ACCESS Pegasus), since a nested apptainer dies with "Failed to set mount
# propagation: Permission denied". Fall back to "vanilla" — PegasusLite launches
# apptainer itself — only if the pool's HTCondor is too old for it.
EXEC_UNIVERSE = "container"

# Last resort if neither universe can run a container: run in the site's native
# environment, which must then already provide the Python deps (see the README).
NO_CONTAINER = False

# Import the generator rather than shelling out to it, so each run is a real
# workflow object we can plan, monitor and analyse in place -- the same shape
# as the other ACCESS Pegasus example notebooks.
sys.path.insert(0, str(WF))
from workflow_generator import build_workflow

# The Pegasus API logs a line per job added; with the NEON fan-out that is 121
# lines of noise before anything interesting. Our own summary stays at INFO.
import logging
logging.getLogger("Pegasus").setLevel(logging.WARNING)

# Every build_workflow() option is a command-line option of
# workflow_generator.py, by its long name with dashes as underscores.
COMMON = dict(config=str(CONFIG), sources=SOURCES,
              start_date=START, end_date=END,
              execution_site_name=SITE, exec_universe=EXEC_UNIVERSE,
              no_container=NO_CONTAINER)

print("mode    :", MODE)
print("site    :", SITE, "|", EXEC_UNIVERSE, "universe")
print("inputs  :", INPUTS)

## 2. Workflow 1 — fetch the inputs

A download-only DAG: every fetcher, nothing computed, staged out **and
registered** into `inputs/`. On a cold run this is the expensive part, and it
is the part you should only ever pay once.

In [ ]:
fetch_wf = None
if RUN_FETCH:
    fetch_wf = build_workflow(mode="fetch", output_dir=str(INPUTS),
                              output=str(WF / "fetch.yml"), **COMMON)
    print("\nfetch DAG : %d jobs -> %s"
          % (len(fetch_wf.wf.jobs), fetch_wf.local_storage_dir))
else:
    n = len(list(INPUTS.glob("*"))) if INPUTS.is_dir() else 0
    print("Skipping the fetch workflow. %s holds %d files." % (INPUTS, n))

In [ ]:
# This cell submits and blocks. Set SUBMIT = False to stay offline — every
# cell above works without a pool, so you can inspect both DAGs first.
SUBMIT = True

if SUBMIT and fetch_wf is not None and HAVE_PLAN:
    fetch_wf.plan_submit()       # pegasus-plan --submit -s SITE -o local
    fetch_wf.wait()              # blocks; Pegasus prints its own progress
    fetch_wf.statistics()
elif fetch_wf is None:
    pass                         # RUN_FETCH is False; inputs/ is already there
elif not HAVE_PLAN:
    print("No Pegasus CLI here. On a submit host:")
    print("  pegasus-plan --submit -s %s -o local fetch.yml" % SITE)
else:
    print("SUBMIT is False — not submitting the fetch workflow.")

## 3. Workflow 2 — the processing run

Build this one **after** the fetch run has finished, not before. Two reasons,
both of which bite silently rather than loudly:

* reuse is resolved **at build time** — the generator scans `inputs/` and
  registers what it finds as replicas, so a directory that is still filling
  registers nothing and every fetch job runs again;
* the catalogs (`replicas.yml`, `sites.yml`, `transformations.yml`) use
  **fixed filenames**, so the last workflow built is the one whose catalogs are
  on disk. Always build the DAG you are about to plan.

In [ ]:
reuse = str(INPUTS) if INPUTS.is_dir() and any(INPUTS.iterdir()) else None
if reuse is None:
    print("WARNING: %s is empty — every fetch job will run again.\n" % INPUTS)

workflow = build_workflow(mode=MODE, reuse_dir=reuse,
                          output=str(WF / (MODE + ".yml")), **COMMON)
DAG = Path(workflow.dagfile)

summary = workflow.reuse_summary()
if summary:
    print("\nreusing %d of %d declared outputs from %s"
          % (summary["n_reused"], summary["n_total_outputs"], INPUTS))
    print("-> Pegasus prunes those producing jobs during planning; everything "
          "downstream of harmonize still runs.")

### What the DAG contains

In [ ]:
import yaml
from collections import Counter

spec = yaml.safe_load(DAG.read_text())
jobs = spec["jobs"]
print(f"{DAG.name}: {len(jobs)} jobs, "
      f"{len(spec.get('jobDependencies', []))} dependency groups\n")

for label, n in Counter(
        j["nodeLabel"].split("_20")[0] for j in jobs).most_common():
    print(f"   {n:4d}  {label}")

**The job count is the point — but only when NEON is in the run.** With a
token, the monthly fetch jobs dominate the DAG and run in parallel; without
one you get the small SCAN-only version, which is correct behaviour rather
than a bug. The fan-out is data-driven: the month list is read from NEON's own
catalogue at generation time, not assumed from a configured lag.

In [ ]:
# Which jobs feed which, for the non-fan-out part of the graph.
by_id = {j["id"]: j["nodeLabel"] for j in jobs}
parents = {}
for d in spec.get("jobDependencies", []):
    for child in d["children"]:
        parents.setdefault(by_id[child], []).append(by_id[d["id"]])

for node in ("harmonize", "build_covariates", "station_similarity",
             "visualize_response", "soil_moisture"):
    if node in parents:
        ps = sorted(parents[node])
        shown = ", ".join(ps[:4]) + (f", (+{len(ps) - 4} more)"
                                     if len(ps) > 4 else "")
        print(f"{node:22s} <- {shown}")

### Render the DAG

In [ ]:
from IPython.display import Image

if shutil.which("pegasus-graphviz"):
    png = WF / f"{MODE}_dag.png"
    rc, log = sh(["pegasus-graphviz", "-f", DAG, "--output", png])
    if png.exists():
        display(Image(filename=str(png)))
    else:
        print("render failed:", log[:400])
else:
    print("pegasus-graphviz not on PATH - skipping DAG render.")
    print("On a submit host:  pegasus-graphviz -f "
          f"{DAG.name} --output {MODE}_dag.png")

## 4. Submit the processing workflow

`plan_submit()` is `pegasus-plan --submit -s SITE -o local`, run against the
catalogs written by the cell above. Outputs stage back to `./output/`.

In [ ]:
if SUBMIT and HAVE_PLAN:
    workflow.plan_submit()
    print("\nrun directory:", workflow.submit_dir or "(planning failed above)")
elif not HAVE_PLAN:
    print("No Pegasus CLI here. On a submit host:")
    print("  pegasus-plan --submit -s %s -o local %s" % (SITE, DAG.name))
else:
    print("SUBMIT is False — not submitting. To run it by hand:")
    print("  pegasus-plan --submit -s %s -o local %s" % (SITE, DAG.name))

## 5. Monitor

`status()` for a snapshot, `wait()` to block until the run finishes,
`statistics()` for per-stage cost afterwards, and `analyze()` for the first
failing job if it does not.

These read the run this notebook planned, and run whenever there is one — they
are gated on the run directory, not on `SUBMIT`. To follow a run submitted
elsewhere,
use `pegasus-status -l <run-dir>` in a terminal — and note that `-w` takes the
poll interval as its own argument (`-w 60 <run-dir>`), so `-w <run-dir>` reads
the directory as an interval and monitors nothing.

In [ ]:
# Gated on the run directory rather than on SUBMIT: that is what
# status/wait/statistics actually need, so a failed plan says so here instead
# of raising further down.
if workflow.submit_dir:
    workflow.status(long=True)
else:
    print("No run to monitor — the workflow was not submitted.")

In [ ]:
if workflow.submit_dir:
    workflow.wait()          # blocks until the DAG finishes
    workflow.statistics()    # per-stage cost from the provenance database
    # workflow.analyze()     # uncomment if it failed: first failing job + logs
else:
    print("Nothing to wait for — the workflow was not submitted.")

### Prefetching without a pool

`./fetch_data.sh` fills the same `inputs/` directory without Pegasus at all —
it asks the generator for the download-only DAG and runs exactly those jobs
itself, so there is no second copy of the fetch logic to drift. Set
`RUN_FETCH = False` above and the notebook picks the directory up.

Measured on a real pool, every row submitted and completed:

| | executable after planning | wall clock |
|---|---|---|
| `--mode all`, nothing prefetched | **186** | 9 min 56 s |
| processing run reusing `inputs/` | **45** | 6 min 1 s |
| reusing a directory that holds *computed products* | **16** | 4 min 2 s |

The last row is the degenerate case: reusing products prunes the jobs that
would have made them, so the run is fast because it does almost nothing. Reuse
`inputs/` to re-run the analysis; reuse a products directory only for a
nowcast, where keeping the covariates, zones and fingerprints is the point.

That reduction *is* the answer to "the map must update as new field data
arrive" — a recurring nowcast is minutes, not a rebuild.

## 6. Inspect the staged results

A pool run stages products to the `local` storage site — `./output/`. Point
`RESULTS` at any directory holding workflow products; `./output/local/` is
where the sibling local-execution notebook writes, so you can preview these
cells without a pool.

In [ ]:
RESULTS = WF / "output"          # pool runs land here


def products(d):
    return {p.name: p for p in sorted(d.glob("*"))
            if p.is_file() and not p.name.startswith("._")} if d.is_dir() else {}


available = products(RESULTS)
if not available:
    # Nothing staged here yet - offer any sibling directory that does have
    # products (e.g. output/local from the local-execution notebook).
    candidates = {d: products(d) for d in sorted(RESULTS.glob("*"))
                  if d.is_dir()} if RESULTS.is_dir() else {}
    candidates = {d: f for d, f in candidates.items() if f}
    print(f"No products in {RESULTS}.")
    for d, f in candidates.items():
        print(f"   {len(f):3d} products in {d}  ->  RESULTS = Path('{d}')")
    if candidates:
        RESULTS, available = next(iter(candidates.items()))
        print(f"\nPreviewing {RESULTS} for the cells below.")

print(f"\n{len(available)} products in {RESULTS}:")
for name, p in available.items():
    print(f"   {p.stat().st_size / 1e6:9.2f} MB  {name}")

In [ ]:
rep_path = available.get("harmonization_report.json")
if rep_path:
    rep = json.loads(rep_path.read_text())
    print("harmonized:", rep["n_observations"], "observations |",
          "duplicates dropped:", rep["duplicates_dropped"])
    display(pd.DataFrame([
        {"source": s, "observations": v["n_observations"],
         "nodes": len(v["nodes"]), "start": v["start"][:10],
         "end": v["end"][:10]}
        for s, v in rep["sources"].items()]))
else:
    print("no harmonization_report.json yet - run the workflow first")

In [ ]:
sm_path = available.get("soil_moisture_points.json")
if sm_path:
    sm = json.loads(sm_path.read_text())
    print(f"as of {sm['as_of']}: {sm['n_nodes']} nodes / "
          f"{sm['n_stations']} stations, {sm['n_stale_nodes']} stale")
    print(f"region mean surface: {sm['region_mean_surface_current']} "
          f"{sm['units']}")
    display(pd.DataFrame(sm["points"])[
        ["node", "current", "current_date", "age_days", "stale", "class"]
    ].sort_values("node"))

In [ ]:
grp = available.get("station_groups.csv")
sim_p = available.get("station_similarity.json")
if grp is not None and sim_p is not None:
    sim = json.loads(sim_p.read_text())
    print("nodes:", sim.get("n_nodes"),
          "| distinct locations:", sim.get("n_distinct_locations"))
    print("clustering:", sim.get("clustering"), "\n")
    display(pd.read_csv(grp)[
        ["node", "depth_cm", "group", "drydown_tau_days",
         "event_delta_per_mm_median", "memory_efolding_days"]
    ].sort_values("node"))
else:
    print("no characterization products - run --mode characterize")

In [ ]:
fig = available.get("station_characterization.png")
display(Image(filename=str(fig))) if fig else print("no figure staged yet")

### The gridded-map products

`zone_stats.json` carries the zone delineation and its validation against
behaviour; `estimation_skill.json` carries the leave-one-station-out numbers
that decide whether the map is worth anything.

In [ ]:
zpath = available.get("zone_stats.json")
if zpath:
    zs = json.loads(Path(zpath).read_text())
    c = zs["clustering"]
    print("k = %s  silhouette = %s  station-free zones: %s" % (
        c["k"], c["silhouette"], zs["station_free_zones"] or "none"))
    v = zs["validation"]
    print("zone-vs-behaviour ARI:", v.get("adjusted_rand_index", v.get("skipped")))
    display(pd.DataFrame([
        {"zone": int(z), "area_ha": d["area_ha"], "nodes": d["n_nodes"],
         "station_free": d["station_free"]}
        for z, d in sorted(zs["zones"].items(), key=lambda kv: int(kv[0]))]))
else:
    print("no zone_stats.json staged yet")

In [ ]:
spath = available.get("estimation_skill.json")
npath = available.get("soil_moisture_now.json")
if npath:
    now = json.loads(Path(npath).read_text())
    print("as of %s | tier %s | %s stations at %s locations" % (
        now["provenance"]["as_of"], now["provenance"]["tier_used"],
        now["n_reporting_stations"], now["n_distinct_locations"]))
    print("site mean %s %s | mean 1-sigma %s | station-free %s ha" % (
        now["summary"]["mean"], now["units"],
        now["summary"]["mean_uncertainty"],
        now["summary"]["area_ha_station_free"]))
if spath:
    sk = json.loads(Path(spath).read_text())
    if not sk.get("skipped"):
        display(pd.DataFrame([
            dict(estimator="zone-anchored", **sk["overall"]),
            dict(estimator="baseline: site mean", **sk["baseline_site_mean"]),
            dict(estimator="baseline: climatology", **sk["baseline_climatology"]),
        ]).set_index("estimator"))
        print("verdict:", sk["verdict"])
else:
    print("no estimation_skill.json staged yet")

In [ ]:
mp = available.get("soil_moisture_map.png")
display(Image(filename=str(mp))) if mp else print("no map figure staged yet")

And the self-contained page — one file, no network access, openable from disk.
This is the artifact to hand to the researcher.

In [ ]:
mh = available.get("soil_moisture_map.html")
if mh:
    print(mh, "%.1f KB" % (Path(mh).stat().st_size / 1024))
    from IPython.display import IFrame
    display(IFrame(src=str(Path(mh).relative_to(WF)), width="100%", height=600))
else:
    print("no soil_moisture_map.html staged yet")

---
## What to look at, and what not to claim

Every run reports its own quality rather than asking you to trust it. Read
these four before drawing any conclusion from the map — they are what changes
when you add stations, change the window, or point the workflow at another
site.

| Where | What it tells you |
|---|---|
| `harmonization_report.json` | which source contributed what, per node — coverage, gaps, rejected rows |
| `response_<station>.json` | the response fingerprints per depth-node, plus QC (coverage, gaps, flatlines, out-of-range days) |
| `station_similarity.json` | behavioural groups and the covariate attribution behind them |
| `zone_stats.json` | the zones and their validation against the *observed* behavioural groups (adjusted Rand index) |
| `estimation_skill.json` | leave-one-station-out RMSE for the estimate **and** for a site-mean and a climatology-only baseline, with a plain-language `verdict` that the HTML page prints above the map |

**How to read them.** Two checks decide whether the map earned its complexity.
First, does the estimate beat *both* baselines in `estimation_skill.json`? Beating
climatology only means the anomaly step works; if it does not also beat the site
mean, the spatial structure is not yet adding information. Second, is the
adjusted Rand index in `zone_stats.json` clearly positive? Zones derived from
covariates should agree with groups derived independently from the observations;
a value near or below zero says the labels have no discriminating power where
they can be checked, usually because the reporting stations are concentrated in
one zone. Both are reported as measured — the workflow does not tune them away,
and a run that fails them is a real result about the station network, not a bug.

**What is physically checkable.** τ should rise with depth (deep soil dries
slowly) and event response should attenuate with depth (the wetting front
weakens going down). A sensor violating both is more likely broken than
interesting. The SSURGO component at each station is an independent read on its
soil.

**What the guards protect.** Attribution across stations rests on the number of
*distinct locations*, not the number of depth-nodes, so the code refuses to fit
below 6 nodes or 3 distinct locations and the figure withholds the
driver-scatter panel on the same test; the tier-2 regression stays off below
`analysis.min_stations_for_regression` distinct locations. A covariate taking
two values across a network will correlate with anything. Nodes whose last
reading is older than `analysis.max_current_age_days` are flagged stale and
excluded rather than published as current. Where the reference site's public
network is sparse these guards fire often — adding stations to
`site_config.json` is what relaxes them.

**Extending this.** The station list, depths, analysis grid and every threshold
above live in `site_config.json`; the observation contract (`timestamp, source,
node, lat, lon, variable, value, unit`) is what a new fetcher has to emit to
join the run. `SPEC.md` documents both.